In [24]:
import pyvisa
import time

In [25]:
rm = pyvisa.ResourceManager()

In [26]:
print(rm.list_resources())

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'TCPIP0::169.254.110.10::inst0::INSTR')


In [28]:
inst = rm.open_resource("TCPIP0::169.254.110.10::inst0::INSTR")
print(inst.query("*IDN?"))

Agilent Technologies, N5183A, MY47420242, A.01.43



In [50]:

inst.write(":FREQ:CW 1000.0MHZ")
inst.write(":POW:LEV -4DBM")

16

In [48]:
inst.write(":OUTP on")

10

In [51]:
inst.write(":OUTP off")

11

In [ ]:
import socket

In [6]:
ip, port = "169.254.110.10", 5025
with socket.create_connection((ip,port),timeout = 5) as s:
    s.sendall(b"*IDN?\n")
    print(s.recv(4096).decode().strip())

Agilent Technologies, N5183A, MY47420242, A.01.43


In [ ]:
import time
import pyvisa

def __init__(visa_name, timeout = 5000):
    rm = pyvisa.ResourceManager("@ivi")
    inst = rm.open_resource(visa_name)
    inst.timeout = timeout
    inst.write("*CLS")
    print(inst.query("*IDN?").strip())
    # return rm, inst

In [24]:
import pyvisa
import time

class Agilent:
    def __init__(self, visa_name, power_limit, freq_limit, timeout = 5000):
        self.set_power_limit(power_limit)
        self.set_freq_limit(freq_limit)
        
        rm = pyvisa.ResourceManager()
        self.inst = rm.open_resource(visa_name)
        self.inst.timeout = timeout
        print(self.inst.query("*IDN?"))
        
        

    def set_power_limit(self, power_limit):
        if power_limit<=15:
            self.power_limit = power_limit
        else:
            raise RuntimeError("the power limit exceeds the limit: 15dBm")

    def set_freq_limit(self, freq_limit):
        if 100e3<=freq_limit<= 40e9:
            self.freq_limit = freq_limit
        else:
            raise RuntimeError("the freq. limit is out of range: [100KHz, 40GHz]")

    def start_output(self):
        self.inst.write(":OUTP on")

    def stop_output(self):
        self.inst.write(":OUTP off")

    def set_frequency(self, frequency):
        if frequency<=self.freq_limit:
            self.inst.write(":FREQ:CW " + str(frequency))
        else:
            raise RuntimeError(f"the freq. exceeds the limit: {self.freq_limit}Hz")

    def set_power(self, power):
        if power<=self.power_limit:
            self.inst.write(":POW:LEV " + str(power) + "DBM")
        else:
            raise RuntimeError(f"the power exceeds the limit: {self.power_limit}dBm")
        

In [47]:
agilent = Agilent("TCPIP0::169.254.110.10::inst0::INSTR",power_limit=0, freq_limit=10e9)

VisaIOError: VI_ERROR_INV_PROT (-1073807239): The protocol specified is invalid.

In [44]:
agilent.set_frequency(1e9)

In [27]:
agilent.set_freq_limit(1e9)

In [38]:
agilent.set_power(-5)

In [34]:
agilent.set_power_limit(0)

In [43]:
agilent.start_output()

In [46]:
agilent.stop_output()